# MiaChat

This will help you getting started with Heroku [chat models](/docs/concepts/chat_models). For detailed documentation of all MiaChat features and configurations head to the [API reference](https://python.langchain.com/api_reference/heroku/chat_models/langchain_heroku.chat_models.MiaChat.html).

## Overview
### Integration details

| Class | Package | Local | Serializable | [JS support](https://js.langchain.com/docs/integrations/chat/heroku) | Package downloads | Package latest |
| :--- | :--- | :---: | :---: |  :---: | :---: | :---: |
| [MiaChat](https://python.langchain.com/api_reference/heroku/chat_models/langchain_heroku.chat_models.MiaChat.html) | [langchain-heroku](https://python.langchain.com/api_reference/heroku/) | ❌ | ❌ | ❌ | ![PyPI - Downloads](https://img.shields.io/pypi/dm/langchain-heroku?style=flat-square&label=%20) | ![PyPI - Version](https://img.shields.io/pypi/v/langchain-heroku?style=flat-square&label=%20) |

### Model features
| [Tool calling](/docs/how_to/tool_calling) | [Structured output](/docs/how_to/structured_output/) | JSON mode | [Image input](/docs/how_to/multimodal_inputs/) | Audio input | Video input | [Token-level streaming](/docs/how_to/chat_streaming/) | Native async | [Token usage](/docs/how_to/chat_token_usage_tracking/) | [Logprobs](/docs/how_to/logprobs/) |
| :---: | :---: | :---: | :---: |  :---: | :---: | :---: | :---: | :---: | :---: |
| ✅ | ❌ | ❌ | ❌ | ❌ | ❌ | ✅ | ❌ | ✅ | ❌ | 

## Setup

To access Heroku models you'll need to create a Heroku account, get an API key, and install the `langchain-heroku` integration package.

### Credentials

To use Heroku's Managed Inference and Agents add-on, you need to:

1. **Install the Heroku CLI and AI Plugin**:
   ```bash
   # Install Heroku CLI if you haven't already
   # Then install the AI plugin
   heroku plugins:install @heroku/plugin-ai
   ```

2. **Create a Heroku App** (if you don't have one):
   ```bash
   heroku create <your-new-app-name>
   ```

3. **Provision an AI Model Resource**:
   ```bash
   # List available models
   heroku ai:models:list
   
   # Create and attach a model to your app
   heroku ai:models:create -a $APP_NAME $MODEL_ID
   
   # Example for Claude 3.5 Sonnet
   heroku ai:models:create -a my-app claude-3-5-sonnet-latest
   ```

4. **Get Your Config Variables**: After attaching a model resource, your app will have three new config variables. You can view them with:
   ```bash
   heroku config -a $APP_NAME
   ```

   The config variables will be:
   - `INFERENCE_KEY`
   - `INFERENCE_MODEL_ID` 
   - `INFERENCE_URL`

5. **Export Environment Variables**: You can export these as environment variables with:
   ```bash
   eval $(heroku config -a $APP_NAME --shell | grep '^INFERENCE_' | tee /dev/tty | sed 's/^/export /')
   ```

Alternatively, you can set them manually in your Python environment:

**Available Models**:
- `claude-3-5-sonnet-latest` - Claude 3.5 Sonnet (recommended for best intelligence)
- `claude-3-5-haiku-latest` - Claude 3.5 Haiku (cost-effective and fast)
- `claude-3-opus-latest` - Claude 3 Opus (most capable)
- `claude-3-sonnet-latest` - Claude 3 Sonnet
- `claude-3-haiku-latest` - Claude 3 Haiku

**Pricing**: Models are billed per token used. See the [Heroku AI pricing page](https://devcenter.heroku.com/articles/heroku-ai-pricing) for current rates.

In [ ]:
import getpass
import os

if not os.getenv("INFERENCE_KEY"):
    os.environ["INFERENCE_KEY"] = getpass.getpass("Enter your Heroku Inference API key: ")
if not os.getenv("INFERENCE_URL"):
    os.environ["INFERENCE_URL"] = input("Enter your Heroku Inference API URL: ")
if not os.getenv("INFERENCE_MODEL_ID"):
    os.environ["INFERENCE_MODEL_ID"] = input("Enter your model ID: ")

If you want to get automated tracing of your model calls you can also set your [LangSmith](https://docs.smith.langchain.com/) API key by uncommenting below:

In [ ]:
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")

### Installation

The LangChain Heroku integration lives in the `langchain-heroku` package:

In [ ]:
%pip install -qU langchain-heroku

## Instantiation

Now we can instantiate our model object and generate chat completions:

In [ ]:
from langchain_heroku import MiaChat

llm = MiaChat(
    model="your-model-id",
    temperature=0.1,
    max_tokens=256,
    timeout=60,
    max_retries=2,
    top_p=0.95,
    streaming=False,
)

## Invocation

You can invoke the model with either a string or a list of messages:

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# Using string input
ai_msg = llm.invoke("Hello! How are you today?")
print(ai_msg.content)

In [ ]:
print(ai_msg.content)

In [ ]:
# Using message list input
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What's the weather like?")
]
ai_msg = llm.invoke(messages)
print(ai_msg.content)

## Streaming

You can enable streaming to get token-level responses:

In [ ]:
# Enable streaming
streaming_llm = MiaChat(
    model="your-model-id",
    temperature=0.1,
    streaming=True
)

# Stream the response
for chunk in streaming_llm.stream("Tell me a short story:"):
    print(chunk.content, end="", flush=True)

## Tool Calling

MiaChat supports tool calling functionality. You can define tools and the model will choose when to use them:

# Define tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA"
                    }
                },
                "required": ["location"]
            }
        }
    }
]

# Create model with tools
tool_llm = MiaChat(
    model="your-model-id",
    tools=tools,
    tool_choice="auto"  # Let the model decide when to use tools
)

# Test tool calling
response = tool_llm.invoke("What's the weather like in New York?")
print(response.content)
if hasattr(response, 'additional_kwargs') and 'tool_calls' in response.additional_kwargs:
    print("\nTool calls:", response.additional_kwargs['tool_calls'])

## Chaining

We can [chain](/docs/how_to/sequence/) our model with a prompt template like so:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate(
    [
        (
            "system",
            "You are a helpful assistant that translates {input_language} to {output_language}.",
        ),
        ("human", "{input}"),
    ]
)

chain = prompt | llm
result = chain.invoke(
    {
        "input_language": "English",
        "output_language": "German",
        "input": "I love programming.",
    }
)
print(result.content)

## API reference

For detailed documentation of all MiaChat features and configurations head to the [API reference](https://python.langchain.com/api_reference/heroku/chat_models/langchain_heroku.chat_models.MiaChat.html)